In [1]:
# CELL 1: KHỞI TẠO VÀ CẤU HÌNH SPARK (CHẾ ĐỘ TIẾT KIỆM TÀI NGUYÊN)
from pyspark.sql import SparkSession

print("Đang khởi tạo SparkSession...")

# Khởi tạo SparkSession kèm các giới hạn tài nguyên khắt khe
spark = (SparkSession.builder 
    .appName("Amazon_Reviews_Processing") 
    .config("spark.executor.instances", "2")   # Chỉ xin 2 cục xử lý
    .config("spark.executor.cores", "2")       # Mỗi cục chỉ dùng 2 core
    .config("spark.executor.memory", "4g")     # Mỗi cục chỉ dùng 4GB RAM
    .config("spark.driver.memory", "4g")       # Cục trung tâm cũng chỉ dùng 4GB RAM
    .config("spark.sql.shuffle.partitions", "800")
    .config("spark.sql.files.maxPartitionBytes", "67108864")
    .getOrCreate()
)

print("Đã khởi tạo SparkSession thành công với cấu hình ít RAM!")

Đang khởi tạo SparkSession...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:19:22 INFO SparkEnv: Registering MapOutputTracker
26/06/14 15:19:22 INFO SparkEnv: Registering BlockManagerMaster
26/06/14 15:19:22 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/14 15:19:22 INFO SparkEnv: Registering OutputCommitCoordinator


Đã khởi tạo SparkSession thành công với cấu hình ít RAM!


In [2]:
# CELL 2: XỬ LÝ BƯỚC 1 VÀ LƯU CHECKPOINT XUỐNG GCS
silver_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/silver/amazon_reviews_cleaned/data/ingest_ts_day=2026-06-14" # Đường dẫn của bạn
checkpoint_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/tmp/silver_checkpoint_step1"

# 1. Đọc dữ liệu từ Silver
df_silver = spark.read.parquet(silver_path)

# 2. Xử lý nhẹ (nếu có: filter null, select cột cần thiết cho Tháp sản phẩm...)
# df_silver_filtered = df_silver.select("item_id", "user_id", "rating", ...)

# 3. GHI XUỐNG GCS (Thay thế hoàn toàn cho df_silver.cache())
print("Đang ghi dữ liệu Bước 1 xuống thư mục tạm trên GCS...")
df_silver.write.mode("overwrite").parquet(checkpoint_path)

print(f"Hoàn thành Bước 1! Dữ liệu đã an toàn tại: {checkpoint_path}")

Đang ghi dữ liệu Bước 1 xuống thư mục tạm trên GCS...


Hoàn thành Bước 1! Dữ liệu đã an toàn tại: gs://amazon-reviews-lakehouse-warehouse/warehouse/tmp/silver_checkpoint_step1


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

checkpoint_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/tmp/silver_checkpoint_step1"

# 1. ĐỌC VÀ CHUẨN HÓA SCHEMA
# Rename cột ngay lập tức để đồng nhất với Entity của Feast Feature Store sau này
df_step2_input = spark.read.parquet(checkpoint_path) \
    .withColumnRenamed("asin", "item_id") \
    .withColumnRenamed("user_id_hashed", "user_id") \
    .withColumnRenamed("text", "review_text")

print("Bắt đầu Bước 2: Xây dựng Đặc trưng Tháp Sản Phẩm (Gold Layer)...")

# 2. TẠO ĐẶC TRƯNG THỐNG KÊ
df_stats = df_step2_input.groupBy("item_id").agg(
    F.count("user_id").alias("review_count"),
    F.round(F.avg("rating"), 2).alias("avg_rating"),
    F.min("timestamp").alias("first_review_time"),
    F.max("timestamp").alias("last_review_time")
)

# 3. TẠO TEXT CONTEXT CHO LLM EMBEDDING
# Giới hạn 3 review mới nhất để tối ưu FinOps (Giảm Token API)
window_spec = Window.partitionBy("item_id").orderBy(F.col("timestamp").desc())

df_ranked_reviews = df_step2_input.withColumn("rank", F.row_number().over(window_spec))

df_latest_reviews = df_ranked_reviews.filter(F.col("rank") <= 3) \
    .groupBy("item_id") \
    .agg(F.collect_list("review_text").alias("recent_reviews")) \
    .withColumn(
        "item_text_context", 
        F.concat_ws(" | ", F.col("recent_reviews"))
    ).drop("recent_reviews")

# 4. JOIN DATA VÀ XỬ LÝ NULL
df_gold_item_features = df_stats.join(df_latest_reviews, on="item_id", how="left")
df_gold_item_features = df_gold_item_features.fillna({"item_text_context": "No review available"})

print("Schema bảng Gold - Item Features sau khi chuẩn hóa:")
df_gold_item_features.printSchema()

# 5. GHI XUỐNG BẢNG GOLD
gold_output_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/item_features/data/"

print("Đang ghi bảng Gold Item Features xuống GCS...")
df_gold_item_features.write \
    .mode("overwrite") \
    .parquet(gold_output_path)

print(f"Hoàn thành Bước 2! Bảng Gold đã sẵn sàng tại: {gold_output_path}")

Bắt đầu Bước 2: Xây dựng Đặc trưng Tháp Sản Phẩm (Gold Layer)...
Schema bảng Gold - Item Features sau khi chuẩn hóa:
root
 |-- item_id: string (nullable = true)
 |-- review_count: long (nullable = false)
 |-- avg_rating: double (nullable = true)
 |-- first_review_time: long (nullable = true)
 |-- last_review_time: long (nullable = true)
 |-- item_text_context: string (nullable = false)

Đang ghi bảng Gold Item Features xuống GCS...


Hoàn thành Bước 2! Bảng Gold đã sẵn sàng tại: gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/item_features/data/
